# Swarini Indic profile pipeline on Modal

This notebook runs teacher generation, canonical CPU audio regeneration, student/GAN training, and model export in separate stages. It supports the Kokoro and IndicVoice profiles in `training/configs/`.

Default selection resumes the existing Hindi run: mount the Volume at **/mnt/hindi-volume**, use profile `training/configs/hindi_kokoro.json`, and run directory `/mnt/hindi-volume/hi_hf_alpha`. To change language or teacher, edit `PROFILE_RELATIVE_PATH` in each code cell that you run. Keep the same choice and Volume across restarts.

**Do not use Run All for staged hardware changes.** Run only the teacher cell on GPU, audio cell on CPU, then student cell on GPU. Modal kernel restarts clear Python state; stage cells restore paths from the mounted Volume and re-extract the uploaded ZIP if needed.

| Stage | Modal GPU | CPU | RAM |
|---|---:|---:|---:|
| Profile text preflight | None | 4–8 cores | 8–16 GiB |
| Teacher generation | A10 24 GB or A100 40 GB | 16 cores | 32–64 GiB |
| Canonical audio regeneration | None | 8–16 cores | 8–16 GiB |
| Student and GAN training | A100 40 GB recommended | 16 cores | 32–64 GiB |

These are planning values, not measured minimums. Modal compute settings are selected in its UI. Keep at least 50 GB free on the persistent Volume. Only the Hindi/Kokoro profile has a completed run; Punjabi and Bengali profiles are templates and must pass CPU phone-coverage preflight and a small pilot first.


## Teachers, language profiles, and research references

The active profile selects Kokoro or IndicVoice teacher loading, voice, G2P backend, symbol map, corpus, and student settings. Hindi Kokoro + `hf_alpha` is the completed result in this checkout. Hindi IndicVoice is wired but experimental. Punjabi and Bengali are unvalidated starter profiles.

At the pinned v1.0 revisions, IndicVoice `indicvoice-v1_0.pth` and Kokoro `kokoro-v1_0.pth` have identical SHA-256 `496dba118d1a58f5f3db2efc88dbdc216e0483fc89fe6e47ee1f2c53f18ad1e4`. This is a frontend/packaging comparison, not independent teacher checkpoints. The student architecture still requires a Kokoro-compatible 24 kHz teacher and validated phones. CPU corpus preparation rejects unknown or out-of-range token IDs before teacher synthesis.

References: [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M), [IndicVoice-82M](https://huggingface.co/Bindkushal/IndicVoice-82M), [Indic G2P](https://github.com/Bindkushal/indic-g2p), [Teacher-in-the-Loop](https://huggingface.co/MithrilMan/teacher-in-the-loop), and [Paradee](https://github.com/sahilmahendrakar/paradee). The pipeline is offline teacher-target distillation; do not call it a full Teacher-in-the-Loop implementation.


## CPU preflight for a new profile

Default Hindi run already has prepared text and teacher shards, so leave the flag off. For another profile, set `PROFILE_RELATIVE_PATH` in this cell and later stage cells. Select CPU resources, set the flag true, and run this cell first. It writes the selected corpus under the mounted Volume, validates G2P phones, and exits before teacher synthesis. Keep the profile and Volume unchanged after kernel restart.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import os
import stat
import subprocess

WORKSPACE = Path.cwd().resolve()
VOLUME_ROOT = Path('/mnt/hindi-volume')  # Existing Modal Volume mount from this notebook.
PROFILE_RELATIVE_PATH = 'training/configs/hindi_kokoro.json'  # Edit for another profile.
RUN_DIR = None  # Resolved from the profile after loading the project.

# Prefer the uploaded bundle so a resource restart cannot leave stale launcher code.
archive = None
for base in (WORKSPACE, *WORKSPACE.parents):
    matches = sorted([*base.glob('Swarini-Indic-training*.zip'), *base.glob('Swarini-Hindi-training*.zip')], key=lambda p: p.stat().st_mtime)
    if matches:
        archive = matches[-1]
        break

if archive is not None:
    extract_root = archive.parent.resolve()
    with ZipFile(archive) as bundle:
        for info in bundle.infolist():
            member = PurePosixPath(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or '..' in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
            destination = (extract_root / Path(*member.parts)).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'ZIP entry escapes workspace: {info.filename}')
        bundle.extractall(extract_root)
    PROJECT_ROOT = extract_root / PurePosixPath(bundle.namelist()[0]).parts[0]
else:
    # A restart may preserve the extracted project but lose the uploaded ZIP.
    PROJECT_ROOT = None
    for base in (WORKSPACE, *WORKSPACE.parents):
        for candidate in (base / 'Swarini-Indic-training', base / 'Swarini-Hindi-training', base):
            if (candidate / 'train_indic.sh').is_file():
                PROJECT_ROOT = candidate.resolve()
                break
        if PROJECT_ROOT:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project files are absent after the Modal restart. Upload the updated '
            'Swarini-Indic-training.zip; the mounted Volume is unaffected.'
        )

if not (PROJECT_ROOT / 'train_indic.sh').is_file():
    raise FileNotFoundError(f'Training launcher missing from {PROJECT_ROOT}')
PROFILE_PATH = (PROJECT_ROOT / PROFILE_RELATIVE_PATH).resolve()
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f'Profile not found: {PROFILE_PATH}')
profile_data = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
RUN_DIR = VOLUME_ROOT / profile_data['run_name']
N_SHARDS = (int(profile_data['corpus'].get('count', 12000)) + 499) // 500
CORPUS_PATH = RUN_DIR / 'corpus' / Path(profile_data['corpus']['path']).name
PROFILE_ENV = {**os.environ, 'SWARINI_PROFILE': str(PROFILE_PATH)}
if not profile_data.get('legacy_compatible', False):
    PROFILE_ENV['SWARINI_CORPUS'] = str(CORPUS_PATH)
runner = (PROJECT_ROOT / 'training' / 'scripts' / 'run_indic.py').read_text(encoding='utf-8')
if '--regen-audio-only' not in runner or '--teacher-only' not in runner or '--prepare-only' not in runner:
    raise RuntimeError(
        'This project bundle is outdated. Upload the updated Swarini-Indic-training.zip '
        'so staged CPU/GPU resume works.'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(
        f'{VOLUME_ROOT} is not mounted. Attach the same existing Modal Volume at this path.'
    )
print(f'Project: {PROJECT_ROOT}')
print(f'Existing Modal Volume: {VOLUME_ROOT}')
print(f'Run directory: {RUN_DIR}')

def load_pipeline_state():
    state_file = RUN_DIR / 'pipeline_state.json'
    if not state_file.is_file():
        return None
    return json.loads(state_file.read_text(encoding='utf-8'))

def check_teacher_corpus():
    state = load_pipeline_state()
    if not state or not state.get('completed', {}).get('generate_teacher'):
        raise RuntimeError('Teacher generation is not marked complete in this Volume/run directory.')
    teacher = RUN_DIR / 'teacher'
    expected = [teacher / f'shard_{i:04d}.pt' for i in range(N_SHARDS)]
    missing = [path.name for path in expected if not path.is_file()]
    if not (teacher / 'manifest.json').is_file() or missing:
        raise RuntimeError(f'Teacher corpus is incomplete; missing manifest/shards: {missing}')
    print(f'Found completed teacher corpus: {N_SHARDS}/{N_SHARDS} shards. Teacher generation will be skipped.')
    return state

RUN_PROFILE_PREFLIGHT = False  # Set True on CPU before a new profile's first GPU run.
if RUN_PROFILE_PREFLIGHT:
    command = [
        'bash', str(PROJECT_ROOT / 'train_indic.sh'),
        '--device', 'cpu', '--prepare-only', '--run-dir', str(RUN_DIR),
    ]
    print('Preparing and validating profile corpus on CPU:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True, env=PROFILE_ENV)
else:
    print('Skipped. For a new profile, run this cell on CPU before teacher generation.')


## Optional: teacher generation for a new profile run

Default run already has all 24 Hindi teacher shards. Leave the flag false for that run. For a new profile, run the CPU preflight cell first, then select GPU resources and set this flag true. It checks the selected profile's persistent run directory and skips teacher work if all expected shards exist.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import os
import stat
import subprocess

WORKSPACE = Path.cwd().resolve()
VOLUME_ROOT = Path('/mnt/hindi-volume')  # Existing Modal Volume mount from this notebook.
PROFILE_RELATIVE_PATH = 'training/configs/hindi_kokoro.json'  # Edit for another profile.
RUN_DIR = None  # Resolved from the profile after loading the project.

# Prefer the uploaded bundle so a resource restart cannot leave stale launcher code.
archive = None
for base in (WORKSPACE, *WORKSPACE.parents):
    matches = sorted([*base.glob('Swarini-Indic-training*.zip'), *base.glob('Swarini-Hindi-training*.zip')], key=lambda p: p.stat().st_mtime)
    if matches:
        archive = matches[-1]
        break

if archive is not None:
    extract_root = archive.parent.resolve()
    with ZipFile(archive) as bundle:
        for info in bundle.infolist():
            member = PurePosixPath(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or '..' in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
            destination = (extract_root / Path(*member.parts)).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'ZIP entry escapes workspace: {info.filename}')
        bundle.extractall(extract_root)
    PROJECT_ROOT = extract_root / PurePosixPath(bundle.namelist()[0]).parts[0]
else:
    # A restart may preserve the extracted project but lose the uploaded ZIP.
    PROJECT_ROOT = None
    for base in (WORKSPACE, *WORKSPACE.parents):
        for candidate in (base / 'Swarini-Indic-training', base / 'Swarini-Hindi-training', base):
            if (candidate / 'train_indic.sh').is_file():
                PROJECT_ROOT = candidate.resolve()
                break
        if PROJECT_ROOT:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project files are absent after the Modal restart. Upload the updated '
            'Swarini-Indic-training.zip; the mounted Volume is unaffected.'
        )

if not (PROJECT_ROOT / 'train_indic.sh').is_file():
    raise FileNotFoundError(f'Training launcher missing from {PROJECT_ROOT}')
PROFILE_PATH = (PROJECT_ROOT / PROFILE_RELATIVE_PATH).resolve()
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f'Profile not found: {PROFILE_PATH}')
profile_data = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
RUN_DIR = VOLUME_ROOT / profile_data['run_name']
N_SHARDS = (int(profile_data['corpus'].get('count', 12000)) + 499) // 500
CORPUS_PATH = RUN_DIR / 'corpus' / Path(profile_data['corpus']['path']).name
PROFILE_ENV = {**os.environ, 'SWARINI_PROFILE': str(PROFILE_PATH)}
if not profile_data.get('legacy_compatible', False):
    PROFILE_ENV['SWARINI_CORPUS'] = str(CORPUS_PATH)
runner = (PROJECT_ROOT / 'training' / 'scripts' / 'run_indic.py').read_text(encoding='utf-8')
if '--regen-audio-only' not in runner or '--teacher-only' not in runner:
    raise RuntimeError(
        'This project bundle is outdated. Upload the updated Swarini-Indic-training.zip '
        'so staged CPU/GPU resume works.'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(
        f'{VOLUME_ROOT} is not mounted. Attach the same existing Modal Volume at this path.'
    )
print(f'Project: {PROJECT_ROOT}')
print(f'Existing Modal Volume: {VOLUME_ROOT}')
print(f'Run directory: {RUN_DIR}')

def load_pipeline_state():
    state_file = RUN_DIR / 'pipeline_state.json'
    if not state_file.is_file():
        return None
    return json.loads(state_file.read_text(encoding='utf-8'))

def check_teacher_corpus():
    state = load_pipeline_state()
    if not state or not state.get('completed', {}).get('generate_teacher'):
        raise RuntimeError('Teacher generation is not marked complete in this Volume/run directory.')
    teacher = RUN_DIR / 'teacher'
    expected = [teacher / f'shard_{i:04d}.pt' for i in range(N_SHARDS)]
    missing = [path.name for path in expected if not path.is_file()]
    if not (teacher / 'manifest.json').is_file() or missing:
        raise RuntimeError(f'Teacher corpus is incomplete; missing manifest/shards: {missing}')
    print(f'Found completed teacher corpus: {N_SHARDS}/{N_SHARDS} shards. Teacher generation will be skipped.')
    return state

START_TEACHER_STAGE = False  # Leave False for your current run; teacher shards already exist.
if START_TEACHER_STAGE:
    if not profile_data.get('legacy_compatible', False):
        if not CORPUS_PATH.is_file() or not CORPUS_PATH.with_suffix('.meta.json').is_file():
            raise RuntimeError('Run the CPU profile preflight cell before teacher generation.')
        corpus_meta = json.loads(CORPUS_PATH.with_suffix('.meta.json').read_text(encoding='utf-8'))
        if corpus_meta.get('profile_id') != profile_data['id'] or corpus_meta.get('count', 0) < profile_data['corpus'].get('count', 12000):
            raise RuntimeError('Profile corpus is incomplete or belongs to another profile. Rerun CPU preflight.')
    state = load_pipeline_state()
    already_done = bool(
        state and state.get('completed', {}).get('generate_teacher')
        and (RUN_DIR / 'teacher' / 'manifest.json').is_file()
        and all((RUN_DIR / 'teacher' / f'shard_{i:04d}.pt').is_file() for i in range(N_SHARDS))
    )
    if already_done:
        print('Teacher corpus already complete. Nothing generated.')
    else:
        gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
        if gpu.returncode:
            raise RuntimeError('Select the A100 GPU profile before teacher generation.')
        subprocess.run([
            'bash', str(PROJECT_ROOT / 'train_indic.sh'),
            '--device', 'cuda', '--teacher-only', '--run-dir', str(RUN_DIR),
        ], cwd=PROJECT_ROOT, check=True, env=PROFILE_ENV)
else:
    print('Skipped. Your existing teacher corpus is already complete.')


## CPU stage: regenerate audio from saved teacher shards

Stop the GPU cell. Select **no GPU, 8–16 CPU cores, 8–16 GiB RAM**, and keep the same Volume and profile. This cell verifies every shard and runs only CPU waveform regeneration. It does not synthesize teacher targets again.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import os
import stat
import subprocess

WORKSPACE = Path.cwd().resolve()
VOLUME_ROOT = Path('/mnt/hindi-volume')  # Existing Modal Volume mount from this notebook.
PROFILE_RELATIVE_PATH = 'training/configs/hindi_kokoro.json'  # Edit for another profile.
RUN_DIR = None  # Resolved from the profile after loading the project.

# Prefer the uploaded bundle so a resource restart cannot leave stale launcher code.
archive = None
for base in (WORKSPACE, *WORKSPACE.parents):
    matches = sorted([*base.glob('Swarini-Indic-training*.zip'), *base.glob('Swarini-Hindi-training*.zip')], key=lambda p: p.stat().st_mtime)
    if matches:
        archive = matches[-1]
        break

if archive is not None:
    extract_root = archive.parent.resolve()
    with ZipFile(archive) as bundle:
        for info in bundle.infolist():
            member = PurePosixPath(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or '..' in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
            destination = (extract_root / Path(*member.parts)).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'ZIP entry escapes workspace: {info.filename}')
        bundle.extractall(extract_root)
    PROJECT_ROOT = extract_root / PurePosixPath(bundle.namelist()[0]).parts[0]
else:
    # A restart may preserve the extracted project but lose the uploaded ZIP.
    PROJECT_ROOT = None
    for base in (WORKSPACE, *WORKSPACE.parents):
        for candidate in (base / 'Swarini-Indic-training', base / 'Swarini-Hindi-training', base):
            if (candidate / 'train_indic.sh').is_file():
                PROJECT_ROOT = candidate.resolve()
                break
        if PROJECT_ROOT:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project files are absent after the Modal restart. Upload the updated '
            'Swarini-Indic-training.zip; the mounted Volume is unaffected.'
        )

if not (PROJECT_ROOT / 'train_indic.sh').is_file():
    raise FileNotFoundError(f'Training launcher missing from {PROJECT_ROOT}')
PROFILE_PATH = (PROJECT_ROOT / PROFILE_RELATIVE_PATH).resolve()
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f'Profile not found: {PROFILE_PATH}')
profile_data = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
RUN_DIR = VOLUME_ROOT / profile_data['run_name']
N_SHARDS = (int(profile_data['corpus'].get('count', 12000)) + 499) // 500
CORPUS_PATH = RUN_DIR / 'corpus' / Path(profile_data['corpus']['path']).name
PROFILE_ENV = {**os.environ, 'SWARINI_PROFILE': str(PROFILE_PATH)}
if not profile_data.get('legacy_compatible', False):
    PROFILE_ENV['SWARINI_CORPUS'] = str(CORPUS_PATH)
runner = (PROJECT_ROOT / 'training' / 'scripts' / 'run_indic.py').read_text(encoding='utf-8')
if '--regen-audio-only' not in runner or '--teacher-only' not in runner:
    raise RuntimeError(
        'This project bundle is outdated. Upload the updated Swarini-Indic-training.zip '
        'so staged CPU/GPU resume works.'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(
        f'{VOLUME_ROOT} is not mounted. Attach the same existing Modal Volume at this path.'
    )
print(f'Project: {PROJECT_ROOT}')
print(f'Existing Modal Volume: {VOLUME_ROOT}')
print(f'Run directory: {RUN_DIR}')

def load_pipeline_state():
    state_file = RUN_DIR / 'pipeline_state.json'
    if not state_file.is_file():
        return None
    return json.loads(state_file.read_text(encoding='utf-8'))

def check_teacher_corpus():
    state = load_pipeline_state()
    if not state or not state.get('completed', {}).get('generate_teacher'):
        raise RuntimeError('Teacher generation is not marked complete in this Volume/run directory.')
    teacher = RUN_DIR / 'teacher'
    expected = [teacher / f'shard_{i:04d}.pt' for i in range(N_SHARDS)]
    missing = [path.name for path in expected if not path.is_file()]
    if not (teacher / 'manifest.json').is_file() or missing:
        raise RuntimeError(f'Teacher corpus is incomplete; missing manifest/shards: {missing}')
    print(f'Found completed teacher corpus: {N_SHARDS}/{N_SHARDS} shards. Teacher generation will be skipped.')
    return state

RUN_CPU_REGENERATION = False  # Set True, then run this cell on the CPU-only profile.
if RUN_CPU_REGENERATION:
    state = check_teacher_corpus()
    if state.get('completed', {}).get('regenerate_audio'):
        print('CPU audio regeneration is already complete. No work needed.')
    else:
        command = [
            'bash', str(PROJECT_ROOT / 'train_indic.sh'),
            '--device', 'cpu', '--regen-audio-only', '--run-dir', str(RUN_DIR),
        ]
        print('Running CPU audio regeneration only:', ' '.join(command))
        subprocess.run(command, cwd=PROJECT_ROOT, check=True, env=PROFILE_ENV)
        print('Done. Regeneration state saved on the Volume.')
else:
    print('Skipped. Set RUN_CPU_REGENERATION = True on the CPU-only profile.')


## GPU stage: resume student and GAN training

After the CPU cell finishes, select an **A100 40 GB, 16 CPU cores, 32–64 GiB RAM**. Kernel restarts. Keep the same profile and Volume. Run only this cell; launcher skips teacher and audio stages, then resumes student training and export checkpoints.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import os
import stat
import subprocess

WORKSPACE = Path.cwd().resolve()
VOLUME_ROOT = Path('/mnt/hindi-volume')  # Existing Modal Volume mount from this notebook.
PROFILE_RELATIVE_PATH = 'training/configs/hindi_kokoro.json'  # Edit for another profile.
RUN_DIR = None  # Resolved from the profile after loading the project.

# Prefer the uploaded bundle so a resource restart cannot leave stale launcher code.
archive = None
for base in (WORKSPACE, *WORKSPACE.parents):
    matches = sorted([*base.glob('Swarini-Indic-training*.zip'), *base.glob('Swarini-Hindi-training*.zip')], key=lambda p: p.stat().st_mtime)
    if matches:
        archive = matches[-1]
        break

if archive is not None:
    extract_root = archive.parent.resolve()
    with ZipFile(archive) as bundle:
        for info in bundle.infolist():
            member = PurePosixPath(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or '..' in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
            destination = (extract_root / Path(*member.parts)).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'ZIP entry escapes workspace: {info.filename}')
        bundle.extractall(extract_root)
    PROJECT_ROOT = extract_root / PurePosixPath(bundle.namelist()[0]).parts[0]
else:
    # A restart may preserve the extracted project but lose the uploaded ZIP.
    PROJECT_ROOT = None
    for base in (WORKSPACE, *WORKSPACE.parents):
        for candidate in (base / 'Swarini-Indic-training', base / 'Swarini-Hindi-training', base):
            if (candidate / 'train_indic.sh').is_file():
                PROJECT_ROOT = candidate.resolve()
                break
        if PROJECT_ROOT:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project files are absent after the Modal restart. Upload the updated '
            'Swarini-Indic-training.zip; the mounted Volume is unaffected.'
        )

if not (PROJECT_ROOT / 'train_indic.sh').is_file():
    raise FileNotFoundError(f'Training launcher missing from {PROJECT_ROOT}')
PROFILE_PATH = (PROJECT_ROOT / PROFILE_RELATIVE_PATH).resolve()
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f'Profile not found: {PROFILE_PATH}')
profile_data = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
RUN_DIR = VOLUME_ROOT / profile_data['run_name']
N_SHARDS = (int(profile_data['corpus'].get('count', 12000)) + 499) // 500
CORPUS_PATH = RUN_DIR / 'corpus' / Path(profile_data['corpus']['path']).name
PROFILE_ENV = {**os.environ, 'SWARINI_PROFILE': str(PROFILE_PATH)}
if not profile_data.get('legacy_compatible', False):
    PROFILE_ENV['SWARINI_CORPUS'] = str(CORPUS_PATH)
runner = (PROJECT_ROOT / 'training' / 'scripts' / 'run_indic.py').read_text(encoding='utf-8')
if '--regen-audio-only' not in runner or '--teacher-only' not in runner:
    raise RuntimeError(
        'This project bundle is outdated. Upload the updated Swarini-Indic-training.zip '
        'so staged CPU/GPU resume works.'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(
        f'{VOLUME_ROOT} is not mounted. Attach the same existing Modal Volume at this path.'
    )
print(f'Project: {PROJECT_ROOT}')
print(f'Existing Modal Volume: {VOLUME_ROOT}')
print(f'Run directory: {RUN_DIR}')

def load_pipeline_state():
    state_file = RUN_DIR / 'pipeline_state.json'
    if not state_file.is_file():
        return None
    return json.loads(state_file.read_text(encoding='utf-8'))

def check_teacher_corpus():
    state = load_pipeline_state()
    if not state or not state.get('completed', {}).get('generate_teacher'):
        raise RuntimeError('Teacher generation is not marked complete in this Volume/run directory.')
    teacher = RUN_DIR / 'teacher'
    expected = [teacher / f'shard_{i:04d}.pt' for i in range(N_SHARDS)]
    missing = [path.name for path in expected if not path.is_file()]
    if not (teacher / 'manifest.json').is_file() or missing:
        raise RuntimeError(f'Teacher corpus is incomplete; missing manifest/shards: {missing}')
    print(f'Found completed teacher corpus: {N_SHARDS}/{N_SHARDS} shards. Teacher generation will be skipped.')
    return state

START_GPU_TRAINING = False  # Set True after switching back to the A100 profile.
if START_GPU_TRAINING:
    state = check_teacher_corpus()
    if not state.get('completed', {}).get('regenerate_audio'):
        raise RuntimeError('CPU regeneration is not complete. Run the CPU stage first.')
    gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
    if gpu.returncode:
        raise RuntimeError('A100 is not visible. Select the A100 GPU profile and rerun this cell.')
    command = [
        'bash', str(PROJECT_ROOT / 'train_indic.sh'),
        '--device', 'cuda', '--run-dir', str(RUN_DIR),
    ]
    print('Resuming student training from the existing Volume:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True, env=PROFILE_ENV)
else:
    print('Skipped. Set START_GPU_TRAINING = True after selecting A100.')


## Download trained model

Run after training completes. It finds the profile's run directory on the mounted Volume and packages FP32, INT8, config, and a sample WAV. The config records the selected language and G2P.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import os
import stat
import subprocess

WORKSPACE = Path.cwd().resolve()
VOLUME_ROOT = Path('/mnt/hindi-volume')  # Existing Modal Volume mount from this notebook.
PROFILE_RELATIVE_PATH = 'training/configs/hindi_kokoro.json'  # Edit for another profile.
RUN_DIR = None  # Resolved from the profile after loading the project.

# Prefer the uploaded bundle so a resource restart cannot leave stale launcher code.
archive = None
for base in (WORKSPACE, *WORKSPACE.parents):
    matches = sorted([*base.glob('Swarini-Indic-training*.zip'), *base.glob('Swarini-Hindi-training*.zip')], key=lambda p: p.stat().st_mtime)
    if matches:
        archive = matches[-1]
        break

if archive is not None:
    extract_root = archive.parent.resolve()
    with ZipFile(archive) as bundle:
        for info in bundle.infolist():
            member = PurePosixPath(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or '..' in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
            destination = (extract_root / Path(*member.parts)).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'ZIP entry escapes workspace: {info.filename}')
        bundle.extractall(extract_root)
    PROJECT_ROOT = extract_root / PurePosixPath(bundle.namelist()[0]).parts[0]
else:
    # A restart may preserve the extracted project but lose the uploaded ZIP.
    PROJECT_ROOT = None
    for base in (WORKSPACE, *WORKSPACE.parents):
        for candidate in (base / 'Swarini-Indic-training', base / 'Swarini-Hindi-training', base):
            if (candidate / 'train_indic.sh').is_file():
                PROJECT_ROOT = candidate.resolve()
                break
        if PROJECT_ROOT:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project files are absent after the Modal restart. Upload the updated '
            'Swarini-Indic-training.zip; the mounted Volume is unaffected.'
        )

if not (PROJECT_ROOT / 'train_indic.sh').is_file():
    raise FileNotFoundError(f'Training launcher missing from {PROJECT_ROOT}')
PROFILE_PATH = (PROJECT_ROOT / PROFILE_RELATIVE_PATH).resolve()
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f'Profile not found: {PROFILE_PATH}')
profile_data = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
RUN_DIR = VOLUME_ROOT / profile_data['run_name']
N_SHARDS = (int(profile_data['corpus'].get('count', 12000)) + 499) // 500
CORPUS_PATH = RUN_DIR / 'corpus' / Path(profile_data['corpus']['path']).name
PROFILE_ENV = {**os.environ, 'SWARINI_PROFILE': str(PROFILE_PATH)}
if not profile_data.get('legacy_compatible', False):
    PROFILE_ENV['SWARINI_CORPUS'] = str(CORPUS_PATH)
runner = (PROJECT_ROOT / 'training' / 'scripts' / 'run_indic.py').read_text(encoding='utf-8')
if '--regen-audio-only' not in runner or '--teacher-only' not in runner:
    raise RuntimeError(
        'This project bundle is outdated. Upload the updated Swarini-Indic-training.zip '
        'so staged CPU/GPU resume works.'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(
        f'{VOLUME_ROOT} is not mounted. Attach the same existing Modal Volume at this path.'
    )
print(f'Project: {PROJECT_ROOT}')
print(f'Existing Modal Volume: {VOLUME_ROOT}')
print(f'Run directory: {RUN_DIR}')

def load_pipeline_state():
    state_file = RUN_DIR / 'pipeline_state.json'
    if not state_file.is_file():
        return None
    return json.loads(state_file.read_text(encoding='utf-8'))

def check_teacher_corpus():
    state = load_pipeline_state()
    if not state or not state.get('completed', {}).get('generate_teacher'):
        raise RuntimeError('Teacher generation is not marked complete in this Volume/run directory.')
    teacher = RUN_DIR / 'teacher'
    expected = [teacher / f'shard_{i:04d}.pt' for i in range(N_SHARDS)]
    missing = [path.name for path in expected if not path.is_file()]
    if not (teacher / 'manifest.json').is_file() or missing:
        raise RuntimeError(f'Teacher corpus is incomplete; missing manifest/shards: {missing}')
    print(f'Found completed teacher corpus: {N_SHARDS}/{N_SHARDS} shards. Teacher generation will be skipped.')
    return state

from IPython.display import FileLink, display
from zipfile import ZIP_DEFLATED

MODEL_DIR = RUN_DIR / 'models'
required_outputs = ['swarini.onnx', 'swarini_int8.onnx', 'config.json']
missing = [name for name in required_outputs if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Model files missing under {MODEL_DIR}: {missing}. Finish/resume GPU training first.')
MODEL_ZIP = WORKSPACE / f"Swarini-{profile_data['language']['name']}-model.zip"
SAMPLE_COPY = WORKSPACE / f"Swarini-{profile_data['language']['name']}-sample-fp32.wav"
sample_command = [
    str(PROJECT_ROOT / 'training' / '.venv' / 'bin' / 'python'), '-m', 'swarini',
    profile_data.get('sample_text', 'नमस्ते, आप कैसे हैं?'), '--model-dir', str(MODEL_DIR), '-o', str(SAMPLE_COPY),
]
subprocess.run(sample_command, cwd=PROJECT_ROOT, check=True, env=PROFILE_ENV)
with ZipFile(MODEL_ZIP, 'w', compression=ZIP_DEFLATED) as archive:
    for name in required_outputs:
        archive.write(MODEL_DIR / name, arcname=f"Swarini-{profile_data['language']['name']}-model/{name}")
    archive.write(SAMPLE_COPY, arcname=f"Swarini-{profile_data['language']['name']}-model/sample_fp32.wav")
print('Models stored persistently at:', MODEL_DIR)
display(FileLink(MODEL_ZIP.name))
display(FileLink(SAMPLE_COPY.name))
